# AIC — Notebook 07: Evaluation & Metrics

Compute offline evaluation metrics (Recall@K, MRR) if you have ground-truth labels.

Can also run an end-to-end pipeline test on sample queries.

In [ ]:
import subprocess, sys, os
GITHUB_REPO = "https://github.com/YOUR_USERNAME/AIC_System.git"
REPO_DIR = "/kaggle/working/AIC_System"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git","clone","--depth","1",GITHUB_REPO,REPO_DIR],check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"pull"],check=True)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",f"{REPO_DIR}/requirements.txt"],check=True)
print('Setup complete.')

In [ ]:
from pathlib import Path
INDEX_DIR   = Path("/kaggle/input/aic-hcmc-indexes")
KF_ROOT     = Path("/kaggle/input/aic-hcmc-data/keyframes/keyframes")
QUERY_FILE  = Path("/kaggle/working/AIC_System/datasets/queries/sample_queries.json")
OUTPUT_DIR  = Path("/kaggle/working/eval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.pipeline.retrieval_pipeline import RetrievalPipeline
import time
print('Loading pipeline (KIS-only, no VLM)...')
t0 = time.time()
pipeline = RetrievalPipeline.from_index_dir(
    index_dir=str(INDEX_DIR),
    clip_model='ViT-B-32',
    enable_vlm=False,  # Quick test without VLM
)
print(f'Ready in {time.time()-t0:.1f}s')

In [ ]:
import json, time
from src.reasoning.query_classifier import QueryClassifier
from src.evaluation.submission_formatter import SubmissionFormatter
from src.evaluation.evaluator import Evaluator
from src.common.enums import QueryType

# Optional: ground truth labels for evaluation
# GT format: {"query_id": {"video_id": "L21_V001", "frame_idx": 1500}}
GT = {}  # ← fill with actual ground truth

queries = json.load(open(QUERY_FILE))
classifier = QueryClassifier()
formatter = SubmissionFormatter(output_dir=str(OUTPUT_DIR))
evaluator = Evaluator(k_values=[1, 5, 10])
results_log = []

for q in queries:
    qid   = str(q.get('query_id', '?'))
    qtype = classifier.classify(q)
    t0    = time.time()
    ev    = pipeline.run(q, query_id=qid)
    latency = time.time() - t0
    if ev:
        if qtype == QueryType.TEXTUAL_KIS: formatter.add_kis(qid, ev)
        elif qtype == QueryType.QA:        formatter.add_qa(qid, ev, ev.metadata.get('answer',''))
        elif qtype == QueryType.TRAKE:
            ts = ev.metadata.get('trake_submission')
            if ts: formatter.add_trake(qid, ts.video_id, {e.event_id: e.frame_idx for e in ts.events})
        results_log.append({'query_id': qid, 'type': qtype.value,
                             'video_id': ev.video_id, 'frame_idx': ev.frame_idx,
                             'latency': f'{latency:.2f}s'})
        if qid in GT:
            evaluator.add_result(qid, ev.video_id, ev.frame_idx,
                                  GT[qid]['video_id'], GT[qid]['frame_idx'], latency=latency)
        print(f'  [{qid}] {ev.video_id} frame={ev.frame_idx} ({latency:.2f}s)')
    else:
        print(f'  [{qid}] No result')

import pandas as pd
pd.DataFrame(results_log)

In [ ]:
formatter.save_kis()
formatter.save_qa()
formatter.save_trake()
formatter.save_json('full_results.json')
if GT:
    evaluator.print_report()
    evaluator.save(str(OUTPUT_DIR / 'eval_report.json'))
else:
    print('No ground truth provided. Submission files saved:')
    for f in sorted(OUTPUT_DIR.glob('*.csv')):
        print(f'  {f.name}: {f.stat().st_size/1024:.1f} KB')